#  Projet Data Viz : Analyse de la Sécheresse et de l'Impact Agricole au Maroc (2018-2025)

**Auteur :** IMANE ATIF

**filière:** DATA SCIENCE


---

## 🎯 1. Introduction et Objectifs

Cette analyse porte sur l'évolution de la santé de la végétation au Maroc, en s'appuyant sur les données satellites de Google Earth Engine (GEE). L'objectif est d'utiliser les outils de la data visualisation pour raconter une histoire basée sur les données et répondre à deux problématiques clés.

### Objectifs du Projet :

1.  **Comparer l'impact de la sécheresse** sur deux grands bassins agricoles : la plaine du **Souss-Massa** (Agadir) et celle du **Haouz** (Marrakech).
2.  **Analyser l'impact de l'agriculture intensive** (type avocats/agrumes) en comparant la santé de ces parcelles à celle du reste de la plaine du Souss.
3.  **Valider les visualisations** en s'appuyant sur les principes du cours (Graphiques en Lignes, Barres, Boîtes à Moustaches et Heatmaps).

---

## 🛠️ 2. Préparation : Chargement des Données

Dans cette étape, nous importons les bibliothèques nécessaires et chargeons nos deux fichiers CSV exportés depuis Google Earth Engine.

In [16]:
import pandas as pd
import plotly.express as px
import plotly.io as pio
pio.templates.default = "plotly_white"


csv_file_1 = 'NDVI_Souss_vs_Haouz_2018-2025.csv'

df1 = pd.read_csv(csv_file_1, skiprows=1, header=None)


df1.columns = ['Date', 'NDVI_Souss', 'NDVI_Haouz']

df1['Date'] = pd.to_datetime(df1['Date'], format='%b %d, %Y')

print("--- Données df1 nettoyées ---")
print(df1.head())
df1.info()


--- Données df1 nettoyées ---
        Date  NDVI_Souss  NDVI_Haouz
0 2018-01-01       0.131       0.254
1 2018-02-01       0.144       0.288
2 2018-03-01       0.198       0.364
3 2018-04-01       0.162       0.336
4 2018-05-01       0.138       0.263
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 93 entries, 0 to 92
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   Date        93 non-null     datetime64[ns]
 1   NDVI_Souss  93 non-null     float64       
 2   NDVI_Haouz  93 non-null     float64       
dtypes: datetime64[ns](1), float64(2)
memory usage: 2.3 KB


In [17]:
df2 = pd.read_csv('NDVI_Avocat_vs_Souss_2018-2025.csv', skiprows=1, header=None)


df2.columns = ['Date', 'NDVI_Avocat', 'NDVI_Reste_Souss']

df2['Date'] = pd.to_datetime(df2['Date'], format='%b %d, %Y')

print("--- Données df2 nettoyées ---")
print(df2.head())
df2.info()

--- Données df2 nettoyées ---
        Date  NDVI_Avocat  NDVI_Reste_Souss
0 2018-01-01        0.449             0.130
1 2018-02-01        0.437             0.147
2 2018-03-01        0.470             0.203
3 2018-04-01        0.393             0.166
4 2018-05-01        0.312             0.141
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 93 entries, 0 to 92
Data columns (total 3 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Date              93 non-null     datetime64[ns]
 1   NDVI_Avocat       93 non-null     float64       
 2   NDVI_Reste_Souss  93 non-null     float64       
dtypes: datetime64[ns](1), float64(2)
memory usage: 2.3 KB


In [19]:
# TRANSFORMER
df1_long = df1.melt(
    id_vars=['Date'],
    value_vars=['NDVI_Souss', 'NDVI_Haouz'],
    var_name='Region',
    value_name='NDVI'
)

In [20]:
df2_long = df2.melt(
    id_vars=['Date'],
    value_vars=['NDVI_Avocat', 'NDVI_Reste_Souss'],
    var_name='Type_Parcelle',
    value_name='NDVI'
)

print("\n--- Données df2 (long) prêtes pour la visualisation ---")
print(df2_long.head())


--- Données df2 (long) prêtes pour la visualisation ---
        Date Type_Parcelle   NDVI
0 2018-01-01   NDVI_Avocat  0.449
1 2018-02-01   NDVI_Avocat  0.437
2 2018-03-01   NDVI_Avocat  0.470
3 2018-04-01   NDVI_Avocat  0.393
4 2018-05-01   NDVI_Avocat  0.312


## 📈 3. Partie 1 : Analyse Comparative (Souss vs. Haouz)

Cette section se concentre sur notre première problématique : la comparaison de l'impact de la sécheresse au niveau macro-régional.

### 3.1 Visualisation 1 : Évolution Temporelle (Graphique en Lignes)

In [21]:
fig1 = px.line(
    df1_long,
    x='Date',
    y='NDVI',
    color='Region',
    title='ÉVOLUTION : NDVI Moyen Mensuel (Souss vs Haouz), 2018-2025',
    labels={'NDVI': 'Indice NDVI Moyen', 'Region': 'Région'},
    color_discrete_map={
        'NDVI_Souss': 'red',
        'NDVI_Haouz': 'blue'
    }
)

fig1.update_layout(hovermode="x unified")
fig1.show()

### Analyse du Graphique 1 : Évolution Souss vs Haouz

Ce premier graphique en lignes (`fig1`) est notre vision d'ensemble. Le **Graphique en Ligne** est l'outil parfait pour visualiser une **évolution dans le temps**.

Nous pouvons en tirer trois observations principales :

1.  **Cycles Saisonniers :** Les deux lignes montrent des "dents de scie" claires, qui correspondent aux cycles saisonniers. Le NDVI (santé de la végétation) atteint un pic en hiver/printemps (saison humide) et un creux en été (saison sèche).

2.  **Comparaison Régionale :** La ligne bleue (`NDVI_Haouz`) est systématiquement plus élevée que la ligne rouge (`NDVI_Souss`). Cela indique que la plaine du Haouz maintient une moyenne de végétation plus dense ou plus saine, potentiellement en raison de sa dépendance aux barrages de l'Atlas, par opposition à une plus grande dépendance aux nappes phréatiques dans le Souss.

3.  **Signature de la Sécheresse :** C'est l'histoire principale. Si l'on compare les pics de 2018-2020 aux pics de 2022-2024, on observe une **tendance claire à la baisse**. La végétation n'atteint plus ses niveaux de santé maximaux, même pendant la "bonne" saison, ce qui est une preuve visuelle directe de l'impact de la sécheresse persistante.

### 3.2 Visualisation 2 : Comparaison du "Stress Estival" (Graphique en Barres)

In [22]:
df1['Annee'] = df1['Date'].dt.year
df1['Mois'] = df1['Date'].dt.month

# Filtrer pour garder que les mois d'été (Juin=6, Juillet=7, Août=8)
df_ete = df1[df1['Mois'].isin([6, 7, 8])]
# Calculer le NDVI moyen de l'été pour chaque année
df_ete_annuel = df_ete.groupby('Annee')[['NDVI_Souss', 'NDVI_Haouz']].mean().reset_index()

df_ete_long = df_ete_annuel.melt(
    id_vars=['Annee'],
    value_vars=['NDVI_Souss', 'NDVI_Haouz'],
    var_name='Region',
    value_name='NDVI_Moyen_Ete'
)

print("\n--- Données 'Été' prêtes pour le Bar Chart ---")
print(df_ete_long.head())

fig2 = px.bar(
    df_ete_long,
    x='Annee',
    y='NDVI_Moyen_Ete',
    color='Region',
    barmode='group', # Met les barres côte-à-côte
    title='COMPARAISON : NDVI Moyen en Été (Stress Hydrique), 2018-2024',
    labels={'NDVI_Moyen_Ete': 'NDVI Moyen (Juin-Août)', 'Annee': 'Année'},
    color_discrete_map={
        'NDVI_Souss': 'red',
        'NDVI_Haouz': 'blue'
    }
)
fig2.show()


--- Données 'Été' prêtes pour le Bar Chart ---
   Annee      Region  NDVI_Moyen_Ete
0   2018  NDVI_Souss        0.107333
1   2019  NDVI_Souss        0.096000
2   2020  NDVI_Souss        0.085333
3   2021  NDVI_Souss        0.101000
4   2022  NDVI_Souss        0.097333


### Analyse du Graphique 2 : Comparaison du "Stress Estival"

Ce **`Graphique en barres`** (`fig2`) est notre première "nouvelle analyse".

Ce graphique nous raconte deux histoires :

1.  **Observation 1 (Confirmation) :**
    Tout comme sur le premier graphique, les barres bleues (`NDVI_Haouz`) sont systématiquement plus hautes que les barres rouges (`NDVI_Souss`). Cela confirme que le Haouz est en moyenne plus "vert", même pendant la saison la plus sèche.

2.  **Observation 2 (L'Histoire Principale) :**
    L'histoire la plus importante est la **tendance générale à la baisse** de la hauteur des barres. Les barres pour 2018 et 2019 sont les plus hautes. À partir de 2021, on observe une chute nette. Les barres pour 2022, 2023 et 2024 sont visiblement les plus basses de la série.

**Conclusion :**
Ce graphique est une preuve très claire et simplifiée de l'impact de la sécheresse. Il démontre que la santé moyenne de la végétation pendant l'été s'est dégradée ces dernières années. Le "Stress Estival" est de plus en plus intense.

## 🥑 4. Partie 2 : Zoom sur l'Agriculture Intensive (Avocat vs. Reste)

Cette section se concentre sur notre deuxième problématique : mesurer la pression de l'agriculture intensive en la comparant à son environnement immédiat.

### 4.1 Visualisation 3 : Évolution Temporelle (Graphique en Lignes)

In [23]:
fig3 = px.line(
    df2_long,
    x='Date',
    y='NDVI',
    color='Type_Parcelle',
    title='ÉVOLUTION : NDVI Avocats vs. Reste de la Plaine du Souss (2018-2025)',
    labels={'NDVI': 'Indice NDVI Moyen', 'Type_Parcelle': 'Type de Parcelle'},
    color_discrete_map={
        'NDVI_Avocat': 'green',
        'NDVI_Reste_Souss': 'orange'
    }
)

fig3.update_layout(hovermode="x unified")

fig3.show()

### Analyse du Graphique 3 : Évolution Avocat vs. Reste du Souss

En utilisant un **`Graphique en Ligne`** , nous comparons l'évolution de nos deux populations *à l'intérieur* de la plaine du Souss.

L'histoire qu'il raconte est une **divergence spectaculaire** :

1.  **Observation 1 (La Ligne Orange - `NDVI_Reste_Souss`) :**
    Cette ligne représente la végétation "normale" et le sol nu de la plaine. Comme attendu, son indice NDVI est **extrêmement bas**, oscillant entre 0.1 et 0.2. Cela montre une région aride, avec une végétation minimale et en grand stress hydrique. C'est notre "ligne de base" de la sécheresse.

2.  **Observation 2 (La Ligne Verte - `NDVI_Avocat`) :**
    Cette ligne représente les parcelles que nous avons identifiées comme étant de l'agriculture intensive (avocats, agrumes). Le contraste est choquant : son NDVI est **systématiquement élevé**, oscillant entre 0.3 et 0.7. Elle montre aussi des cycles saisonniers (croissance au printemps), mais son point le plus *bas* (en été) est toujours plus élevé que le point le plus *haut* de la ligne orange.

**Conclusion (L'Histoire Principale) :**
Ce graphique est la **preuve visuelle** de l'impact de l'agriculture intensive. Il n'est pas "normal" que la ligne verte soit si saine alors que son environnement immédiat (la ligne orange) est si sec.

La seule explication est une **irrigation massive et constante**. Ces parcelles "vertes" survivent à la sécheresse en consommant d'énormes volumes d'eau (très probablement par **pompage dans les nappes phréatiques**), épuisant ainsi une ressource vitale tout en masquant visuellement l'état de sécheresse réel de la région.

### 4.2 Visualisation 4 : Comparaison des Distributions (Boîte à Moustaches)

In [24]:
fig4 = px.box(
    df2_long,
    x='Type_Parcelle',
    y='NDVI',
    color='Type_Parcelle',
    title='DISTRIBUTION : NDVI Avocats vs. Reste du Souss (2018-2025)',
    labels={'NDVI': 'Indice NDVI Moyen', 'Type_Parcelle': 'Type de Parcelle'},
    color_discrete_map={
        'NDVI_Avocat': 'green',
        'NDVI_Reste_Souss': 'orange'
    }
)

fig4.show()

### Analyse du Graphique 4 : Distribution (Boîte à Moustaches)

Ce graphique (`fig4`) utilise la **`Boîte à moustaches (Boxplot)`**, un outil recommande pour :
> "comparer une même caractéristique entre différentes populations"

Ce visuel nous donne une vue statistique de nos deux populations sur l'ensemble de la période 2018-2025.

L'histoire est encore plus claire qu'avant :

1.  **La Boîte Orange (`NDVI_Reste_Souss`) :**
    Cette boîte est très basse et écrasée. Elle montre que 50% de toutes les observations (le "cœur" des données) se situent dans une plage très faible, autour de 0.1. C'est la signature statistique d'un désert ou d'une plaine en état de sécheresse permanent.

2.  **La Boîte Verte (`NDVI_Avocat`) :**
    Cette boîte est située beaucoup plus haut. 50% de ses observations se situent dans une plage saine, entre ~0.3 et ~0.6.

**Conclusion (La Preuve Statistique) :**
La conclusion la plus puissante est l'**absence totale de chevauchement (overlap)** entre les deux boîtes (les Intervalles Interquartiles).

Cela signifie que **50% du temps, les parcelles d'avocats les moins saines (le bas de la boîte verte) sont TOUJOURS en meilleure santé que 75% des parcelles du "Reste du Souss" (le haut de la boîte orange)**. C'est la confirmation statistique qu'il s'agit de deux mondes complètement différents, artificiellement séparés par l'irrigation intensive.

### 4.3 Visualisation 5 : Analyse des "Patterns" de Sécheresse (Heatmaps)

In [25]:
df2['Annee'] = df2['Date'].dt.year
df2['Mois'] = df2['Date'].dt.month_name()

heatmap_reste = df2.pivot_table(
    index='Mois',
    columns='Annee',
    values='NDVI_Reste_Souss'
)

mois_ordre = ['January', 'February', 'March', 'April', 'May', 'June',
              'July', 'August', 'September', 'October', 'November', 'December']
heatmap_reste = heatmap_reste.reindex(mois_ordre)

print("\n--- Données prêtes pour la Heatmap 1 ---")
print(heatmap_reste.head())

fig5_reste = px.imshow(
    heatmap_reste,
    labels=dict(x="Année", y="Mois", color="NDVI Moyen"),
    title='HEATMAP A (Motif) : NDVI du "Reste de la plaine du Souss"',
    color_continuous_scale='RdYlGn'
)

fig5_reste.show()


heatmap_avocat = df2.pivot_table(
    index='Mois',
    columns='Annee',
    values='NDVI_Avocat'
)
heatmap_avocat = heatmap_avocat.reindex(mois_ordre)

fig5_avocat = px.imshow(
    heatmap_avocat,
    labels=dict(x="Année", y="Mois", color="NDVI Moyen"),
    title='HEATMAP B (Motif) : NDVI des "Parcelles Avocats/Intensif"',
    color_continuous_scale='RdYlGn'
)

fig5_avocat.show()


--- Données prêtes pour la Heatmap 1 ---
Annee      2018   2019   2020   2021   2022   2023   2024   2025
Mois                                                            
January   0.130  0.196  0.103  0.123  0.126  0.128  0.095  0.075
February  0.147  0.187  0.131  0.241  0.110  0.151  0.112  0.083
March     0.203  0.151  0.112  0.227  0.109  0.164  0.119  0.103
April     0.166  0.137  0.110  0.167  0.155  0.135  0.124  0.122
May       0.141  0.120  0.101  0.129  0.139  0.112  0.108  0.110


### Analyse du Graphique 5 : Heatmaps (Les "Patterns" de Sécheresse)


En les plaçant côte à côte, l'histoire de notre projet devient une évidence visuelle.

#### 1. Heatmap A : `NDVI du "Reste de la plaine du Souss"`

* **Ce que nous voyons :** Cette carte est dominée par le **rouge** et l'**orange** (NDVI faible).
* **Le Pattern Horizontal :** On voit un "cycle" clair. Les mois d'été (Juin, Juillet, Août) sont systématiquement rouge foncé, confirmant le "stress estival".
* **Le Pattern Vertical :** En regardant les colonnes de 2022, 2023 et 2024, on voit que même les mois d'hiver/printemps (Jan-Avr) sont d'un vert plus pâle ou plus jaune que lors des années 2018-2019.
* **Conclusion :** C'est le **portrait de la sécheresse**. C'est à quoi ressemble une plaine aride qui souffre d'un manque d'eau chronique.

#### 2. Heatmap B : `NDVI des "Parcelles Avocats/Intensif"`

* **Ce que nous voyons :** Cette carte est dominée par le **vert vif** (NDVI élevé).
* **Le Pattern :** Il n'y a **presque pas de rouge**. Même en plein été (Juin, Juillet), alors que l'autre heatmap est en "souffrance" maximale, les parcelles d'avocats restent d'un vert sain.
* **Conclusion :** C'est le **portrait de l'irrigation intensive**. Cette heatmap est complètement déconnectée de la réalité climatique de la région (montrée par la Heatmap A). Elle prouve que ces parcelles ne dépendent pas de la pluie, mais d'un apport en eau artificiel et constant.



## 🏁 Conclusion Générale du Projet

Ce Projet a commencé par une question simple : "Peut-on *voir* l'impact de la sécheresse ?" La réponse est oui, et l'analyse révèle une histoire bien plus complexe.

1.  **L'Évolution (Graphique en Lignes) :** Nos graphiques en lignes ont confirmé une **tendance à la baisse** de la santé globale de la végétation dans les deux régions du Souss et du Haouz, prouvant l'effet de la sécheresse sur le long terme.

2.  **La Comparaison (Graphique en Barres) :** Notre analyse du "Stress Estival" a **quantifié** cette tendance, montrant que les étés sont de plus en plus "secs" (NDVI plus bas) ces dernières années.

3.  **La Preuve (Graphiques Avocat) :** L'analyse de la "Partie 2" est la conclusion principale. Les **boîtes à moustaches** et les **heatmaps** ont démontré, statistiquement et visuellement, que l'agriculture intensive (avocats/agrumes) crée des "îlots" de verdure artificielle.

Ces parcelles restent en parfaite santé en pompant massivement dans les ressources en eau (nappes phréatiques), alors même que le reste de la plaine (leur environnement direct) est dans un état de sécheresse visible et permanent. Ce projet démontre la puissance de la data visualisation pour raconter des histoires complexes et révéler les pressions invisibles sur notre environnement.